# SessionTransformer Training — Synerise E-commerce Dataset

Trains an autoregressive session transformer with three factored cross-entropy losses:
- **action loss** — predict next event type (page_visit, add_to_cart, …, EOS)
- **item loss** — predict next SKU at item-bearing positions only
- **temporal loss** — predict inter-event time delta (64 log-spaced bins)

Cross-session history is encoded via learned attention pooling.
Training uses teacher forcing with mixed-precision (AMP) and cosine LR annealing.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import time
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional

import joblib
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset

# ── Configuration ─────────────────────────────────────────────────────────────
OUTPUT_DIR         = Path("/home/mofu/code/thesis/pipeline/output")
CLEAN_PARQUET      = OUTPUT_DIR / "events_clean.parquet"
SKU2IDX_PATH       = OUTPUT_DIR / "sku2idx.joblib"
MODEL_DIR          = OUTPUT_DIR / "models"

TRAIN_CUTOFF       = "2022-11-15"
VAL_CUTOFF         = "2022-12-01"

VOCAB_K            = 630052   # top-K SKUs covering ~90% of training item events

N_TEMPORAL_BINS    = 64
TEMPORAL_MIN_S     = 1        # minimum inter-event delta (seconds)
TEMPORAL_MAX_S     = 1800     # 30 min × 60 — max intra-session delta

HISTORY_WINDOW     = 50       # max past events used as cross-session context

TRAIN_EPOCHS       = 10
TRAIN_BATCH_SIZE   = 128
TRAIN_MAX_LENGTH   = 50       # max events per session (including EOS)
TRAIN_LR           = 0.0003   # AdamW learning rate
TRAIN_D_MODEL      = 256      # transformer hidden dimension
TRAIN_N_LAYERS     = 4        # number of decoder layers
TRAIN_N_HEADS      = 8        # attention heads
TRAIN_MAX_SESSIONS = None     # None = all sessions; set int for quick tests

MODEL_DIR.mkdir(parents=True, exist_ok=True)

## Dataset

In [2]:
def build_sku2idx(df_train_skus: pd.Series) -> dict:
    """
    Map the top-(VOCAB_K-1) most frequent SKUs to indices 1..VOCAB_K-1.
    Index 0 is reserved for padding / unknown items.
    """
    top_skus = (
        df_train_skus
        .dropna()
        .astype(int)
        .value_counts()
        .head(VOCAB_K - 1)
        .index
        .tolist()
    )
    return {sku: idx + 1 for idx, sku in enumerate(top_skus)}


def get_sku2idx(df_train_skus: pd.Series = None) -> dict:
    """Load cached sku2idx or build and cache it."""
    if SKU2IDX_PATH.exists():
        return joblib.load(SKU2IDX_PATH)
    if df_train_skus is None:
        raise RuntimeError(
            f"sku2idx not found at {SKU2IDX_PATH}. "
            "Pass df_train_skus to build it."
        )
    mapping = build_sku2idx(df_train_skus)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(mapping, SKU2IDX_PATH)
    print(f"  sku2idx saved → {SKU2IDX_PATH}  ({len(mapping):,} SKUs)")
    return mapping

In [3]:
class SessionDataset(Dataset):
    """
    One item = one session.

    __getitem__ returns:
        events  : LongTensor [T]   action indices (ACTION2IDX)
        items   : LongTensor [T]   sku+1 for item-bearing events; 0 otherwise
        deltas  : LongTensor [T]   temporal bin indices (first event = bin 0)
        length  : int              actual session length T (before padding)
        history : dict | None      cross-session context from prior sessions of
                                   the same user (keys: events, items, deltas,
                                   length); None for first session of a user

    Sessions shorter than min_length are excluded.
    Sessions longer than max_length are truncated (most-recent events kept).
    """

    def __init__(
        self,
        parquet_path=None,
        split: str = "train",
        max_length: int = 50,
        min_length: int = 2,
        max_sessions: int = None,
        history_window: int = HISTORY_WINDOW,
    ):
        self.max_length     = max_length
        self.min_length     = min_length
        self._history_window = history_window

        path = str(parquet_path or CLEAN_PARQUET)

        df = pl.read_parquet(
            path,
            columns=["client_id", "timestamp", "event_type", "session_id", "sku"],
        ).sort(["session_id", "timestamp"])

        # Session-start times for split filtering — in Rust
        train_cut = pd.Timestamp(TRAIN_CUTOFF)
        val_cut   = pd.Timestamp(VAL_CUTOFF)
        session_starts = df.group_by("session_id").agg(
            pl.col("timestamp").min().alias("session_start")
        )
        df = df.join(session_starts, on="session_id")

        train_filter = pl.col("session_start") < train_cut
        if split == "train":
            df = df.filter(train_filter)
        elif split == "val":
            df = df.filter((pl.col("session_start") >= train_cut) & (pl.col("session_start") < val_cut))
        elif split == "test":
            df = df.filter(pl.col("session_start") >= val_cut)
        else:
            raise ValueError(f"split must be 'train', 'val', or 'test'; got '{split}'")

        # Build / load sku→embedding-index mapping (train split only, cached to disk).
        # When split != 'train', df contains only val/test rows so train_filter yields
        # nothing — rely on the cache built during the first train run instead.
        if SKU2IDX_PATH.exists():
            sku2idx = get_sku2idx()
        elif split == "train":
            sku2idx = get_sku2idx(df.select("sku").to_series().to_pandas())
        else:
            raise RuntimeError(
                f"sku2idx cache not found at {SKU2IDX_PATH}. "
                "Run SessionDataset(split='train') first to build the vocabulary."
            )

        #  Vectorised feature computation (stays in Rust/numpy), replaced with Polars because Pandas enjoys
        #  jumping back to Python code instead of staying in C

        # Action indices via polars replace
        df = df.with_columns(
            pl.col("event_type")
            .replace(ACTION2IDX, default=0)
            .cast(pl.Int32)
            .alias("action_idx")
        )

        # Item indices via join against sku2idx mapping
        sku2idx_df = pl.DataFrame(
            {"sku": list(sku2idx.keys()), "item_idx": list(sku2idx.values())},
            schema={"sku": pl.Int64, "item_idx": pl.Int32},
        )
        df = (
            df.with_columns(pl.col("sku").cast(pl.Int64, strict=False))
            .join(sku2idx_df, on="sku", how="left")
            .with_columns(pl.col("item_idx").fill_null(0))
        )

        # Temporal delta (seconds) within session — vectorised diff
        df = df.with_columns(
            pl.col("timestamp").diff().over("session_id")
            .dt.total_microseconds()
            .truediv(1_000_000)   # → seconds as float
            .fill_null(0.0)
            .alias("delta_s")
        )

        # Convert delta_s → bin index via numpy searchsorted (one array op)
        delta_s_np = df["delta_s"].to_numpy()
        delta_s_clipped = np.clip(delta_s_np, TEMPORAL_MIN_S, TEMPORAL_MAX_S)
        bin_indices = np.minimum(
            np.searchsorted(BIN_EDGES[1:], delta_s_clipped),
            N_TEMPORAL_BINS - 1,
        ).astype(np.int32)
        # First event in each session: delta=0 → bin 0
        first_event_mask = df["delta_s"].to_numpy() == 0.0
        bin_indices[first_event_mask & (delta_s_np == 0.0)] = 0
        df = df.with_columns(pl.Series("delta_bin", bin_indices))

        # --- Group into sessions and build tensor records ---
        sessions_df = (
            df.group_by("session_id", maintain_order=True)
            .agg(
                pl.col("client_id").first().alias("client_id"),
                pl.col("action_idx").alias("actions"),
                pl.col("item_idx").alias("items"),
                pl.col("delta_bin").alias("deltas"),
                pl.len().alias("n"),
            )
            .filter(pl.col("n") >= min_length)
        )
        if max_sessions is not None:
            sessions_df = sessions_df.head(max_sessions)

        # Single Python pass: build tensors from polars lists.
        # EOS is appended as the final token so the model learns session termination
        # from real boundaries (PvA §5.3). One slot is reserved from max_length.
        self._sessions: List[Dict] = []
        for row in sessions_df.iter_rows(named=True):
            n_raw   = min(int(row["n"]), max_length - 1)   # reserve one slot for EOS
            actions = row["actions"][-n_raw:] + [EOS_IDX]
            items   = row["items"][-n_raw:]   + [0]
            deltas  = row["deltas"][-n_raw:]  + [0]
            n       = n_raw + 1

            self._sessions.append({
                "client_id": int(row["client_id"]),
                "events":    torch.tensor(actions, dtype=torch.long),
                "items":     torch.tensor(items,   dtype=torch.long),
                "deltas":    torch.tensor(deltas,  dtype=torch.long),
                "length":    n,
            })

        # Build per-user ordered index (session_id is time-based so order is preserved)
        user_idx: dict = defaultdict(list)
        for i, s in enumerate(self._sessions):
            user_idx[s["client_id"]].append(i)
        self._user_session_indices: Dict[int, List[int]] = dict(user_idx)

        # Precompute each session's position within its user's session list
        self._session_pos_in_user: Dict[int, int] = {}
        for user_sessions in self._user_session_indices.values():
            for pos, sess_idx in enumerate(user_sessions):
                self._session_pos_in_user[sess_idx] = pos

    def __len__(self) -> int:
        return len(self._sessions)

    def __getitem__(self, idx: int) -> Dict:
        sess        = self._sessions[idx]
        cid         = sess["client_id"]
        pos         = self._session_pos_in_user[idx]
        prior_idxs  = self._user_session_indices[cid][:pos]

        if prior_idxs:
            prior_e = torch.cat([self._sessions[i]["events"] for i in prior_idxs])
            prior_i = torch.cat([self._sessions[i]["items"]  for i in prior_idxs])
            prior_d = torch.cat([self._sessions[i]["deltas"] for i in prior_idxs])
            H = min(len(prior_e), self._history_window)
            history = {
                "events": prior_e[-H:],
                "items":  prior_i[-H:],
                "deltas": prior_d[-H:],
                "length": H,
            }
        else:
            history = None

        return {**sess, "history": history}

In [4]:
def collate_fn(batch: List[Dict]) -> Dict:
    """
    Pad a list of variable-length sessions to the longest in the batch.

    Returns dict:
        events              : LongTensor [B, T_max]
        items               : LongTensor [B, T_max]
        deltas              : LongTensor [B, T_max]
        lengths             : LongTensor [B]
        tgt_key_padding_mask: BoolTensor [B, T_max-1]  True = padded position
        history             : dict | None
            events          : LongTensor [B, H_max]
            items           : LongTensor [B, H_max]
            deltas          : LongTensor [B, H_max]
            padding_mask    : BoolTensor [B, H_max]   True = padded position
    """
    max_len = max(s["length"] for s in batch)
    B = len(batch)

    ev = torch.zeros(B, max_len, dtype=torch.long)
    it = torch.zeros(B, max_len, dtype=torch.long)
    dl = torch.zeros(B, max_len, dtype=torch.long)
    ln = torch.zeros(B, dtype=torch.long)

    for i, s in enumerate(batch):
        L = s["length"]
        ev[i, :L] = s["events"]
        it[i, :L] = s["items"]
        dl[i, :L] = s["deltas"]
        ln[i]     = L

    # Padding mask for transformer input (positions T-1 since forward shifts by 1)
    mask_len = max_len - 1
    pad_mask = torch.zeros(B, mask_len, dtype=torch.bool)
    for i, s in enumerate(batch):
        L = s["length"]
        if L < mask_len:
            pad_mask[i, L:] = True

    # History: pad to max history length in batch; None if no session has history
    histories = [s.get("history") for s in batch]
    if any(h is not None for h in histories):
        max_H = max((h["length"] for h in histories if h is not None), default=0)
        h_e    = torch.zeros(B, max_H, dtype=torch.long)
        h_i    = torch.zeros(B, max_H, dtype=torch.long)
        h_d    = torch.zeros(B, max_H, dtype=torch.long)
        h_mask = torch.ones(B, max_H, dtype=torch.bool)   # True = padding
        for b, h in enumerate(histories):
            if h is not None:
                H = h["length"]
                h_e[b, :H]    = h["events"]
                h_i[b, :H]    = h["items"]
                h_d[b, :H]    = h["deltas"]
                h_mask[b, :H] = False   # real positions are not masked
        history_out: Optional[Dict] = {
            "events": h_e, "items": h_i, "deltas": h_d, "padding_mask": h_mask,
        }
    else:
        history_out = None

    return {
        "events":               ev,
        "items":                it,
        "deltas":               dl,
        "lengths":              ln,
        "tgt_key_padding_mask": pad_mask,
        "history":              history_out,
    }

In [5]:
class InteractionGenerator:
    """
    Convenience wrapper: constructs SessionDataset + torch DataLoader.
    Used by the training script.

    Attributes
    ----------
    dataset : SessionDataset
    loader  : torch.utils.data.DataLoader
    """

    def __init__(
        self,
        split: str = "train",
        batch_size: int = 64,
        max_length: int = 50,
        min_length: int = 2,
        num_workers: int = 4,
        parquet_path=None,
        max_sessions: int = None,
        history_window: int = HISTORY_WINDOW,
    ):
        self.dataset = SessionDataset(
            parquet_path=parquet_path,
            split=split,
            max_length=max_length,
            min_length=min_length,
            max_sessions=max_sessions,
            history_window=history_window,
        )
        self.loader = DataLoader(
            self.dataset,
            batch_size=batch_size,
            shuffle=(split == "train"),
            collate_fn=collate_fn,
            num_workers=num_workers,
            pin_memory=True,
        )

    def __len__(self) -> int:
        return len(self.loader)

## Model: Vocabulary & Temporal Bins

In [6]:
ACTION_TYPES = [
    "page_visit",        # 0
    "search_query",      # 1
    "add_to_cart",       # 2
    "remove_from_cart",  # 3
    "product_buy",       # 4
    "EOS",               # 5
]
ACTION2IDX: Dict[str, int] = {a: i for i, a in enumerate(ACTION_TYPES)}
IDX2ACTION: Dict[int, str] = {i: a for a, i in ACTION2IDX.items()}
EOS_IDX = ACTION2IDX["EOS"]
ITEM_BEARING_IDX = {ACTION2IDX["add_to_cart"], ACTION2IDX["remove_from_cart"], ACTION2IDX["product_buy"]}
N_ACTIONS = len(ACTION_TYPES)

In [7]:
BIN_EDGES = np.logspace(
    np.log10(TEMPORAL_MIN_S),
    np.log10(TEMPORAL_MAX_S),
    N_TEMPORAL_BINS + 1,
)


def seconds_to_bin(secs: float) -> int:
    secs = float(np.clip(secs, TEMPORAL_MIN_S, TEMPORAL_MAX_S))
    return min(int(np.searchsorted(BIN_EDGES[1:], secs)), N_TEMPORAL_BINS - 1)


def bin_to_seconds(bin_idx: int) -> float:
    lo = BIN_EDGES[bin_idx]
    hi = BIN_EDGES[min(bin_idx + 1, N_TEMPORAL_BINS)]
    return float(np.sqrt(lo * hi))   # geometric midpoint

## Model: Architecture Components

In [8]:
class TransitionConstraintMask:
    """
    Pre-sampling mask: zeroes logits for illegal next actions.
    Applied inside ActionHead at inference time.

    valid_transitions : action_str → set[action_str]
                        EOS is always allowed from any state.

    Stores a [N_ACTIONS+1, N_ACTIONS] matrix where row N_ACTIONS is the
    unconstrained row (all ones), used when last_action_idx is None / -1.
    """

    def __init__(self, valid_transitions: dict):
        # Default: all actions allowed (row = all ones)
        mat = torch.ones(N_ACTIONS + 1, N_ACTIONS)
        for src_str, tgt_set in valid_transitions.items():
            src_idx = ACTION2IDX.get(src_str)
            if src_idx is None:
                continue
            row = torch.zeros(N_ACTIONS)
            for tgt_str in tgt_set:
                tgt_idx = ACTION2IDX.get(tgt_str)
                if tgt_idx is not None:
                    row[tgt_idx] = 1.0
            row[EOS_IDX] = 1.0   # always allow EOS
            mat[src_idx] = row
        self._allowed_matrix = mat   # [N_ACTIONS+1, N_ACTIONS]

    def apply(
        self,
        logits: torch.Tensor,           # [..., N_ACTIONS]
        last_action_idx: Optional[int],
    ) -> torch.Tensor:
        """Return logits with -inf at disallowed positions. No-op if no constraint."""
        if last_action_idx is None:
            return logits
        row = self._allowed_matrix[last_action_idx].bool().to(logits.device)
        return logits.masked_fill(~row, float("-inf"))

    def apply_batch(
        self,
        logits: torch.Tensor,           # [B, N_ACTIONS]
        last_action_idxs: torch.Tensor, # [B] long — -1 means unconstrained
    ) -> torch.Tensor:
        """Vectorised batch version. -1 in last_action_idxs → no constraint."""
        idx = torch.where(
            last_action_idxs < 0,
            torch.full_like(last_action_idxs, N_ACTIONS),   # unconstrained row
            last_action_idxs,
        )
        allowed = self._allowed_matrix.to(logits.device)[idx].bool()  # [B, N_ACTIONS]
        return logits.masked_fill(~allowed, float("-inf"))

In [9]:
class CrossSessionHistory(nn.Module):
    """
    Encodes up to window_H past events as cross-attention memory.

    Receives pre-embedded history (embedded by the parent SessionTransformer
    using its shared embedding tables) and produces a [B, 1, d_model] context
    vector via learned attention pooling.
    """

    def __init__(self, window_H: int, d_model: int):
        super().__init__()
        self.window_H   = window_H
        self.d_model    = d_model
        self.pool_query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.attn       = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True)

    def encode(
        self,
        hist_emb: torch.Tensor,
        key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        hist_emb         : [B, H, d_model]  — pre-embedded history events
        key_padding_mask : [B, H] bool, True = padded position (ignored in attn)
        Returns          : [B, 1, d_model]  — pooled context for cross-attention
        """
        B   = hist_emb.size(0)
        q   = self.pool_query.expand(B, -1, -1)
        ctx, _ = self.attn(q, hist_emb, hist_emb,
                           key_padding_mask=key_padding_mask.float() if key_padding_mask is not None else None)
        return ctx

In [10]:
class ActionHead(nn.Module):
    """
    Predicts next action (5 event types + EOS).
    TransitionConstraintMask is applied at inference time (last_action_idx != None).
    During training last_action_idx is None → no masking → standard cross-entropy.
    """

    def __init__(self, d_model: int, n_actions: int, valid_transitions: dict):
        super().__init__()
        self.fc              = nn.Linear(d_model, n_actions)
        self.constraint_mask = TransitionConstraintMask(valid_transitions)

    def forward(
        self,
        h_t: torch.Tensor,                      # [B, d_model] or [B, T, d_model]
        last_action_idx: Optional[int] = None,  # scalar; None during training
    ) -> torch.Tensor:                           # [B, n_actions] or [B, T, n_actions]
        logits = self.fc(h_t)
        return self.constraint_mask.apply(logits, last_action_idx)

In [11]:
class ItemHead(nn.Module):
    """
    Predicts next SKU. Conditioned on action_t via a small action embedding
    (avoids coupling the full 630K-embedding to this head).
    Top-k is enforced at inference time in SessionTransformer.infer().
    """

    def __init__(self, d_model: int, vocab_size: int, top_k: int = 100):
        super().__init__()
        self.fc          = nn.Linear(d_model, vocab_size)
        self.top_k       = top_k
        self.action_cond = nn.Embedding(N_ACTIONS, d_model)

    def forward(
        self,
        h_t: torch.Tensor,       # [B, d_model] or [B, T, d_model]
        action_t: torch.Tensor,  # [B] or [B, T] — action indices
    ) -> torch.Tensor:           # [B, vocab_size] or [B, T, vocab_size]
        cond = h_t + self.action_cond(action_t)
        return self.fc(cond)

In [12]:
class TemporalHead(nn.Module):
    """
    Predicts inter-event delta_t as a categorical over N_TEMPORAL_BINS log bins.

    Fully factored conditioning:
        delta_{t+1} ~ TemporalHead(h_t, action_{t+1}, item_emb_{t+1})

    item_emb_t is the pre-embedded item vector from SessionTransformer.item_emb,
    passed in rather than re-embedded here to avoid a duplicate 630k-row table.

    Training: action_t and item_emb_t use gold next tokens (teacher-forced).
    Inference: they use the sampled outputs of the upstream heads.
    """

    def __init__(self, d_model: int, n_bins: int = N_TEMPORAL_BINS):
        super().__init__()
        self.fc          = nn.Linear(d_model, n_bins)
        self.n_bins      = n_bins
        self.action_cond = nn.Embedding(N_ACTIONS, d_model)

    def forward(
        self,
        h_t: torch.Tensor,           # [B, d_model] or [B, T, d_model]
        action_t: torch.Tensor,      # [B] or [B, T]  — action indices
        item_emb_t: torch.Tensor,    # [B, d_model] or [B, T, d_model] — pre-embedded item
    ) -> torch.Tensor:               # [B, n_bins] or [B, T, n_bins]
        cond = h_t + self.action_cond(action_t) + item_emb_t
        return self.fc(cond)

In [13]:
class SessionTransformer(nn.Module):
    """
    Full autoregressive session generator.

    Training
    --------
    loss = CE(action_logits, tgt_actions)
         + CE(item_logits[item-bearing positions], tgt_items[item-bearing])
         + CE(temporal_logits, tgt_deltas)

    Inference
    ---------
    Use SessionTransformer.infer(client_id, sku, start_dt, history).

    Parameters
    ----------
    vocab_size        : unique SKUs — VOCAB_K = 630,052
    n_actions         : 5 event types + EOS = 6
    d_model           : transformer hidden dimension (default 256)
    n_layers          : decoder layers (default 4)
    n_heads           : attention heads (default 8)
    n_temporal_bins   : delta_t bins (default 64)
    window_H          : cross-session history window size
    valid_transitions : dict action_str → set[action_str] for constraint mask
    """

    def __init__(
        self,
        vocab_size: int,
        n_actions: int = N_ACTIONS,
        d_model: int = 256,
        n_layers: int = 4,
        n_heads: int = 8,
        n_temporal_bins: int = N_TEMPORAL_BINS,
        window_H: int = 50,
        valid_transitions: dict = None,
    ):
        super().__init__()
        self.d_model    = d_model
        self.vocab_size = vocab_size

        # Shared embedding tables
        self.event_emb = nn.Embedding(n_actions, d_model)
        self.item_emb  = nn.Embedding(vocab_size + 1, d_model, padding_idx=0)
        self.delta_emb = nn.Embedding(n_temporal_bins, d_model)
        self.pos_emb   = nn.Embedding(512, d_model)

        # Causal decoder backbone (cross-attention to history memory)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)

        # Cross-session history encoder
        self.history = CrossSessionHistory(window_H, d_model)

        # Prediction heads
        self.action_head   = ActionHead(d_model, n_actions, valid_transitions or {})
        self.item_head     = ItemHead(d_model, vocab_size, top_k=100)
        self.temporal_head = TemporalHead(d_model, n_temporal_bins)

    # Training forward (teacher-forced, gold conditioning)

    def forward(
        self,
        events: torch.Tensor,                         # [B, T] action indices
        items: torch.Tensor,                          # [B, T] item indices (0=no item)
        deltas: torch.Tensor,                         # [B, T] temporal bin indices
        history: Optional[dict] = None,               # keys: events/items/deltas → [B, H]
        tgt_key_padding_mask: Optional[torch.Tensor] = None,  # [B, T-1] True=pad
        item_mask: Optional[torch.Tensor] = None,     # [B, T-1] bool — item positions only
    ):
        """
        Teacher-forced training pass.

        Input tokens  : t = 0 .. T-2
        Target tokens : t = 1 .. T-1   (shifted by 1)

        Returns
        -------
        action_logits   : [B, T-1, n_actions]
        item_logits     : [M, vocab_size]   if item_mask provided (M = item_mask.sum())
                          [B, T-1, vocab_size]  otherwise (may OOM for large vocabs)
        temporal_logits : [B, T-1, n_bins]

        item_mask should be precomputed in the training loop (valid & tgt_is_item)
        to avoid allocating the full [B, T-1, vocab_size] tensor.
        """
        in_e  = events[:, :-1]   # [B, T-1] — model input
        in_i  = items[:, :-1]
        in_d  = deltas[:, :-1]
        tgt_e = events[:, 1:]    # [B, T-1] — gold next actions (for conditioning)
        tgt_i = items[:, 1:]     # [B, T-1] — gold next items

        B, T = in_e.shape
        device = in_e.device

        pos = torch.arange(T, device=device).unsqueeze(0)   # [1, T]
        x   = (
            self.event_emb(in_e)
            + self.item_emb(in_i)
            + self.delta_emb(in_d)
            + self.pos_emb(pos)
        )  # [B, T, d_model]

        memory   = self._encode_history(history, B, device)                         # [B, >=1, d_model]
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(T, device=device) # [T, T]

        h = self.decoder(
            x, memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
        )  # [B, T, d_model]

        action_logits = self.action_head(h)                      # [B, T, n_actions]

        # Pre-embed gold next items (shared table) for item + temporal heads
        tgt_item_emb = self.item_emb(tgt_i)                     # [B, T, d_model]

        temporal_logits = self.temporal_head(h, tgt_e, tgt_item_emb)  # [B, T, n_bins]

        if item_mask is not None:
            # Sparse: only compute item logits at item-bearing positions → [M, vocab_size]
            h_item      = h[item_mask]       # [M, d_model]
            tgt_e_item  = tgt_e[item_mask]   # [M]
            item_logits = self.item_head(h_item, tgt_e_item)   # [M, vocab_size]
        else:
            item_logits = self.item_head(h, tgt_e)             # [B, T, vocab_size]

        return action_logits, item_logits, temporal_logits

    # Autoregressive inference

    @torch.no_grad()
    def infer(
        self,
        client_id: int,
        sku: int,
        start_dt,
        history: Optional[List[dict]] = None,
        max_steps: int = 50,
        temperature: float = 1.0,
    ) -> List[dict]:
        """
        Generate one session autoregressively.

        Parameters
        ----------
        client_id   : user id (from SimpleIdentitySampler)
        sku         : seed item (0-indexed, from SimpleIdentitySampler)
        start_dt    : session start time (datetime or pd.Timestamp)
        history     : past event dicts for cross-session conditioning
        max_steps   : max events before forced stop
        temperature : sampling temperature

        Returns
        -------
        List of event dicts: {client_id, event_type, sku, timestamp}
        """
        self.eval()
        device = next(self.parameters()).device

        if not isinstance(start_dt, pd.Timestamp):
            start_dt = pd.Timestamp(start_dt)

        # Encode cross-session history → cross-attention memory
        if history:
            hist_emb = self._history_dicts_to_emb(history, device)  # [1, H, d_model]
            memory   = self.history.encode(hist_emb)                 # [1, 1, d_model]
        else:
            memory = torch.zeros(1, 1, self.d_model, device=device)

        # Load sku2idx mapping for correct embedding lookup (trained with remapped indices)
        sku2idx: dict = getattr(self, "_sku2idx", {})
        idx2sku: dict = getattr(self, "_idx2sku", {})

        # Seed token: primes the generator with the seed item identity.
        # Treated as a synthetic "exposure" at t=0; not emitted in output.
        seed_item_idx = sku2idx.get(int(sku), 0) if sku2idx else min(int(sku) + 1, self.vocab_size - 1)
        cur_e = [ACTION2IDX["page_visit"]]
        cur_i = [seed_item_idx]
        cur_d = [0]

        events_out: List[dict] = []
        last_action_idx: Optional[int] = None
        current_dt = start_dt

        for _ in range(max_steps):
            e_t = torch.tensor([cur_e], dtype=torch.long, device=device)
            i_t = torch.tensor([cur_i], dtype=torch.long, device=device)
            d_t = torch.tensor([cur_d], dtype=torch.long, device=device)

            T   = e_t.size(1)
            pos = torch.arange(T, device=device).unsqueeze(0)
            x   = (
                self.event_emb(e_t)
                + self.item_emb(i_t)
                + self.delta_emb(d_t)
                + self.pos_emb(pos)
            )

            tgt_mask = nn.Transformer.generate_square_subsequent_mask(T, device=device)
            h   = self.decoder(x, memory, tgt_mask=tgt_mask)  # [1, T, d_model]
            h_t = h[:, -1, :]                                  # [1, d_model]

            # Action
            a_logits = self.action_head(h_t, last_action_idx) / temperature
            a_probs  = F.softmax(a_logits, dim=-1)
            a_idx    = int(torch.multinomial(a_probs, 1).item())

            if a_idx == EOS_IDX:
                break

            # Item (item-bearing actions only)
            a_t = torch.tensor([a_idx], dtype=torch.long, device=device)
            if a_idx in ITEM_BEARING_IDX:
                i_logits = self.item_head(h_t, a_t) / temperature
                k        = min(self.item_head.top_k, self.vocab_size)
                topk_v, topk_ids = torch.topk(i_logits, k, dim=-1)
                i_probs  = F.softmax(topk_v, dim=-1)
                chosen   = int(torch.multinomial(i_probs, 1).item())
                i_idx    = int(topk_ids[0, chosen].item())
            else:
                i_idx = 0   # padding for non-item-bearing events

            # Temporal bin → seconds
            i_t_s      = torch.tensor([i_idx], dtype=torch.long, device=device)
            i_emb_t    = self.item_emb(i_t_s)                           # [1, d_model]
            d_logits   = self.temporal_head(h_t, a_t, i_emb_t) / temperature
            d_probs  = F.softmax(d_logits, dim=-1)
            bin_idx  = int(torch.multinomial(d_probs, 1).item())
            delta_s  = bin_to_seconds(bin_idx)

            current_dt = current_dt + pd.Timedelta(seconds=delta_s)
            # Decode item index → raw SKU via idx2sku (inverse of sku2idx mapping)
            if i_idx > 0:
                sku_out = idx2sku.get(i_idx, i_idx - 1) if idx2sku else i_idx - 1
            else:
                sku_out = None

            events_out.append({
                "client_id":  client_id,
                "event_type": IDX2ACTION[a_idx],
                "sku":        sku_out,
                "timestamp":  current_dt,
            })

            cur_e.append(a_idx)
            cur_i.append(i_idx)
            cur_d.append(bin_idx)
            last_action_idx = a_idx

        return events_out

    # Batched autoregressive inference, dont ask me how this works, I don't know either

    @torch.no_grad()
    def infer_batch(
        self,
        batch_inputs: List[tuple],  # [(client_id, sku, start_dt, history), ...]
        max_steps: int = 50,
        temperature: float = 1.0,
    ) -> List[List[dict]]:
        """
        Generate B sessions in parallel.

        All sessions share a batch dimension and advance one token per step.
        Done sessions (EOS emitted) are masked out and their tokens are ignored.

        Parameters are
        ----------
        batch_inputs : list of (client_id, sku, start_dt, history) tuples
        max_steps    : max events per session before forced stop
        temperature  : sampling temperature

        Returns
        -------
        List[List[dict]] — one event-dict list per input
        """
        self.eval()
        device  = next(self.parameters()).device
        B       = len(batch_inputs)
        sku2idx = getattr(self, "_sku2idx", {})
        idx2sku = getattr(self, "_idx2sku", {})

        # Per-session history → memory [B, 1, d_model] This fucking sucks
        memories = []
        for _, _, _, history in batch_inputs:
            if history:
                hist_emb = self._history_dicts_to_emb(history, device)  # [1, H, d_model]
                mem      = self.history.encode(hist_emb)                 # [1, 1, d_model]
            else:
                mem = torch.zeros(1, 1, self.d_model, device=device)
            memories.append(mem)
        memory = torch.cat(memories, dim=0)  # [B, 1, d_model]

        # Seed tokens
        seed_items = [
            sku2idx.get(int(sku), 0) if sku2idx else min(int(sku) + 1, self.vocab_size - 1)
            for _, sku, _, _ in batch_inputs
        ]
        cur_e = torch.tensor([[ACTION2IDX["page_visit"]] for _ in range(B)],
                             dtype=torch.long, device=device)   # [B, 1]
        cur_i = torch.tensor([[s] for s in seed_items],
                             dtype=torch.long, device=device)   # [B, 1]
        cur_d = torch.zeros(B, 1, dtype=torch.long, device=device)  # [B, 1]

        done             = torch.zeros(B, dtype=torch.bool, device=device)
        last_action_idxs = torch.full((B,), -1, dtype=torch.long, device=device)
        # Ok
        events_out  = [[] for _ in range(B)]
        current_dts = []
        for _, _, start_dt, _ in batch_inputs:
            if not isinstance(start_dt, pd.Timestamp):
                start_dt = pd.Timestamp(start_dt)
            current_dts.append(start_dt)

        for _ in range(max_steps):
            if done.all():
                break

            T   = cur_e.size(1)
            pos = torch.arange(T, device=device).unsqueeze(0)   # [1, T] broadcast
            x   = (
                self.event_emb(cur_e)
                + self.item_emb(cur_i)
                + self.delta_emb(cur_d)
                + self.pos_emb(pos)
            )  # [B, T, d_model]

            tgt_mask = nn.Transformer.generate_square_subsequent_mask(T, device=device)
            h   = self.decoder(x, memory, tgt_mask=tgt_mask)  # [B, T, d_model]
            h_t = h[:, -1, :]                                  # [B, d_model]

            #  Action
            a_logits = self.action_head.fc(h_t) / temperature  # [B, N_ACTIONS]
            a_logits = self.action_head.constraint_mask.apply_batch(a_logits, last_action_idxs)
            a_probs  = F.softmax(a_logits, dim=-1)
            a_idxs   = torch.multinomial(a_probs, 1).squeeze(1)  # [B]

            newly_done = (a_idxs == EOS_IDX) | done

            # Item (item-bearing, non-done sessions only)
            is_item = torch.zeros(B, dtype=torch.bool, device=device)
            for idx in ITEM_BEARING_IDX:
                is_item |= (a_idxs == idx)
            is_item &= ~done

            i_idxs = torch.zeros(B, dtype=torch.long, device=device)
            if is_item.any():
                h_item   = h_t[is_item]       # [M, d_model]
                a_item   = a_idxs[is_item]    # [M]
                i_logits = self.item_head(h_item, a_item) / temperature  # [M, vocab_size]
                k        = min(self.item_head.top_k, self.vocab_size)
                topk_v, topk_ids = torch.topk(i_logits, k, dim=-1)      # [M, k]
                i_probs  = F.softmax(topk_v, dim=-1)
                chosen   = torch.multinomial(i_probs, 1).squeeze(1)     # [M]
                i_idxs[is_item] = topk_ids.gather(1, chosen.unsqueeze(1)).squeeze(1)
            # This is bad but works for now
            # Temporal
            i_emb_t  = self.item_emb(i_idxs)  # [B, d_model]
            d_logits = self.temporal_head(h_t, a_idxs, i_emb_t) / temperature
            d_probs  = F.softmax(d_logits, dim=-1)
            bin_idxs = torch.multinomial(d_probs, 1).squeeze(1)  # [B]

            # Collect outputs
            for b in range(B):
                if done[b] or a_idxs[b] == EOS_IDX:
                    continue
                a_idx   = int(a_idxs[b].item())
                i_idx   = int(i_idxs[b].item())
                bin_idx = int(bin_idxs[b].item())
                current_dts[b] = current_dts[b] + pd.Timedelta(seconds=bin_to_seconds(bin_idx))
                sku_out = None
                if i_idx > 0:
                    sku_out = idx2sku.get(i_idx, i_idx - 1) if idx2sku else i_idx - 1
                events_out[b].append({
                    "client_id":  batch_inputs[b][0],
                    "event_type": IDX2ACTION[a_idx],
                    "sku":        sku_out,
                    "timestamp":  current_dts[b],
                })

            # Grow sequence tensors (all sessions, incl. done — keeps shape uniform)
            cur_e = torch.cat([cur_e, a_idxs.unsqueeze(1)], dim=1)
            cur_i = torch.cat([cur_i, i_idxs.unsqueeze(1)], dim=1)
            cur_d = torch.cat([cur_d, bin_idxs.unsqueeze(1)], dim=1)

            last_action_idxs = torch.where(done, last_action_idxs, a_idxs)
            done = newly_done

        return events_out

    # Checkpoint I/O

    def save(self, path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "state_dict": self.state_dict(),
            "config": {
                "vocab_size":      self.vocab_size,
                "d_model":         self.d_model,
                "n_actions":       self.action_head.fc.out_features,
                "n_layers":        len(self.decoder.layers),
                "n_heads":         self.decoder.layers[0].self_attn.num_heads,
                "n_temporal_bins": self.temporal_head.n_bins,
                "window_H":        self.history.window_H,
            },
        }, path)
        print(f"  SessionTransformer saved → {path}")

    @classmethod
    def load(
        cls,
        path,
        valid_transitions: dict = None,
        device: str = "cpu",
    ) -> "SessionTransformer":
        path       = Path(path)
        checkpoint = torch.load(path, map_location=device, weights_only=False)
        cfg        = checkpoint["config"]
        model      = cls(
            vocab_size        = cfg["vocab_size"],
            d_model           = cfg["d_model"],
            n_actions         = cfg["n_actions"],
            n_layers          = cfg["n_layers"],
            n_heads           = cfg["n_heads"],
            n_temporal_bins   = cfg["n_temporal_bins"],
            window_H          = cfg["window_H"],
            valid_transitions = valid_transitions or {},
        )
        model.load_state_dict(checkpoint["state_dict"])
        model.to(device)
        model.eval()
        print(f"  SessionTransformer loaded ← {path}")
        return model

    # Private helpers

    def _encode_history(
        self,
        history: Optional[dict],
        batch_size: int,
        device: torch.device,
    ) -> torch.Tensor:
        """
        Produce cross-attention memory from history tensors (training) or
        a zero dummy tensor when history is None.

        history : dict with keys 'events', 'items', 'deltas' → [B, H] tensors
        Returns : [B, 1, d_model]
        """
        if history is None:
            return torch.zeros(batch_size, 1, self.d_model, device=device)

        h_e  = history["events"].to(device)   # [B, H]
        h_i  = history["items"].to(device)
        h_d  = history["deltas"].to(device)
        h_pad = history.get("padding_mask")
        if h_pad is not None:
            h_pad = h_pad.to(device)
        H   = h_e.size(1)
        pos = torch.arange(H, device=device).unsqueeze(0)
        hist_emb = (
            self.event_emb(h_e)
            + self.item_emb(h_i)
            + self.delta_emb(h_d)
            + self.pos_emb(pos)
        )  # [B, H, d_model]
        return self.history.encode(hist_emb, key_padding_mask=h_pad)

    def _history_dicts_to_emb(
        self,
        history: List[dict],
        device: torch.device,
    ) -> torch.Tensor:
        """
        Convert a list of event dicts (past session history) to an embedded
        tensor for CrossSessionHistory.encode(). Truncates to window_H.

        Returns [1, H, d_model].
        """
        history  = history[-self.history.window_H :]
        sku2idx  = getattr(self, "_sku2idx", {})
        h_e, h_i, h_d = [], [], []
        prev_ts = None
        for ev in history:
            a_str = ev.get("event_type", "page_visit")
            h_e.append(ACTION2IDX.get(a_str, 0))
            sku = ev.get("sku")
            if sku is not None:
                h_i.append(sku2idx.get(int(sku), 0) if sku2idx else min(int(sku) + 1, self.vocab_size))
            else:
                h_i.append(0)
            ts = ev.get("timestamp")
            if ts is not None and prev_ts is not None:
                delta_s = (pd.Timestamp(ts) - pd.Timestamp(prev_ts)).total_seconds()
                h_d.append(seconds_to_bin(max(delta_s, TEMPORAL_MIN_S)))
            else:
                h_d.append(0)
            prev_ts = ts

        H   = len(h_e)
        pos = torch.arange(H, device=device).unsqueeze(0)
        e_t = torch.tensor([h_e], dtype=torch.long, device=device)
        i_t = torch.tensor([h_i], dtype=torch.long, device=device)
        d_t = torch.tensor([h_d], dtype=torch.long, device=device)
        return (
            self.event_emb(e_t)
            + self.item_emb(i_t)
            + self.delta_emb(d_t)
            + self.pos_emb(pos)
        )  # [1, H, d_model]

## Training

In [14]:
def build_target_mask(lengths, T_out, device):
    """
    Boolean mask [B, T-1] — True at positions that are REAL targets (not padding).
    For a session of length L, real output positions are 0..L-2
    (predicting tokens 1..L-1 from inputs 0..L-2).
    """
    positions = torch.arange(T_out, device=device).unsqueeze(0)   # [1, T_out]
    return positions < (lengths.to(device).unsqueeze(1) - 1)       # [B, T_out]

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

print(f"\nLoading training dataset (max_sessions={TRAIN_MAX_SESSIONS}) ...")
gen = InteractionGenerator(
    split="train",
    batch_size=TRAIN_BATCH_SIZE,
    max_length=TRAIN_MAX_LENGTH,
    num_workers=0,
    max_sessions=TRAIN_MAX_SESSIONS,
)
print(f"  {len(gen.dataset):,} training sessions | {len(gen):,} batches/epoch")

print(f"\nLoading validation dataset ...")
val_gen = InteractionGenerator(
    split="val",
    batch_size=TRAIN_BATCH_SIZE,
    max_length=TRAIN_MAX_LENGTH,
    num_workers=0,
    max_sessions=None,
)
print(f"  {len(val_gen.dataset):,} val sessions | {len(val_gen):,} batches")

Device: cuda

Loading training dataset (max_sessions=1000) ...


/tmp/ipykernel_2094142/1197623458.py:75: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  .replace(ACTION2IDX, default=0)


  1,000 training sessions | 8 batches/epoch

Loading validation dataset ...


  524,497 val sessions | 4,098 batches


In [16]:
model = SessionTransformer(
    vocab_size=VOCAB_K,
    d_model=TRAIN_D_MODEL,
    n_layers=TRAIN_N_LAYERS,
    n_heads=TRAIN_N_HEADS,
    n_temporal_bins=N_TEMPORAL_BINS,
    window_H=HISTORY_WINDOW,
    valid_transitions={},     # constraint mask not used during training
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

Model parameters: 327,864,170


In [17]:
optimizer = AdamW(model.parameters(), lr=TRAIN_LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS * len(gen), eta_min=1e-6)
scaler    = GradScaler("cuda")

print(f"Config: epochs={TRAIN_EPOCHS}, batch={TRAIN_BATCH_SIZE}, "
      f"d_model={TRAIN_D_MODEL}, layers={TRAIN_N_LAYERS}, heads={TRAIN_N_HEADS}, "
      f"lr={TRAIN_LR}")

Config: epochs=1, batch=128, d_model=256, layers=4, heads=8, lr=0.0003


In [18]:
best_loss = float("inf")

for epoch in range(1, TRAIN_EPOCHS + 1):
    model.train()
    epoch_action = epoch_item = epoch_temporal = 0.0
    epoch_batches = 0
    t0 = time.time()

    for batch in gen.loader:
        events   = batch["events"].to(device)
        items    = batch["items"].to(device)
        deltas   = batch["deltas"].to(device)
        pad_mask = batch["tgt_key_padding_mask"].to(device)
        lengths  = batch["lengths"]
        history  = batch.get("history")
        if history is not None:
            history = {k: v.to(device) for k, v in history.items()}

        B, T  = events.shape
        T_out = T - 1

        tgt_actions = events[:, 1:]
        tgt_items   = items[:, 1:]
        tgt_deltas  = deltas[:, 1:]

        valid = build_target_mask(lengths, T_out, device)

        tgt_is_item = torch.zeros_like(tgt_actions, dtype=torch.bool)
        for idx in ITEM_BEARING_IDX:
            tgt_is_item |= (tgt_actions == idx)
        item_valid = valid & tgt_is_item

        with autocast("cuda"):
            action_logits, item_logits, temporal_logits = model(
                events, items, deltas,
                history=history,
                tgt_key_padding_mask=pad_mask,
                item_mask=item_valid,
            )

            a_logits    = action_logits[valid]
            action_loss = F.cross_entropy(a_logits, tgt_actions[valid])

            if item_valid.any():
                item_loss = F.cross_entropy(item_logits, tgt_items[item_valid])
            else:
                item_loss = torch.zeros(1, device=device).squeeze()

            d_logits      = temporal_logits[valid]
            temporal_loss = F.cross_entropy(d_logits, tgt_deltas[valid])

            loss = action_loss + item_loss + temporal_loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_action   += action_loss.item()
        epoch_item     += item_loss.item()
        epoch_temporal += temporal_loss.item()
        epoch_batches  += 1

        if epoch_batches % max(1, len(gen) // 10) == 0:
            print(
                f"  [{epoch}/{TRAIN_EPOCHS}] batch {epoch_batches}/{len(gen)} "
                f"| act={epoch_action/epoch_batches:.3f} "
                f"item={epoch_item/epoch_batches:.3f} "
                f"temp={epoch_temporal/epoch_batches:.3f}",
                end="\r",
            )

    elapsed  = time.time() - t0
    avg_loss = (epoch_action + epoch_item + epoch_temporal) / epoch_batches
    print(
        f"\nEpoch {epoch}/{TRAIN_EPOCHS} | "
        f"action={epoch_action/epoch_batches:.4f} "
        f"item={epoch_item/epoch_batches:.4f} "
        f"temporal={epoch_temporal/epoch_batches:.4f} "
        f"| total={avg_loss:.4f} | {elapsed:.1f}s"
    )

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    val_action = val_item = val_temporal = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_gen.loader:
            events   = batch["events"].to(device)
            items    = batch["items"].to(device)
            deltas   = batch["deltas"].to(device)
            pad_mask = batch["tgt_key_padding_mask"].to(device)
            lengths  = batch["lengths"]
            history  = batch.get("history")
            if history is not None:
                history = {k: v.to(device) for k, v in history.items()}

            B, T  = events.shape
            T_out = T - 1
            tgt_actions = events[:, 1:]
            tgt_items   = items[:, 1:]
            tgt_deltas  = deltas[:, 1:]
            valid       = build_target_mask(lengths, T_out, device)

            tgt_is_item = torch.zeros_like(tgt_actions, dtype=torch.bool)
            for idx in ITEM_BEARING_IDX:
                tgt_is_item |= (tgt_actions == idx)
            item_valid = valid & tgt_is_item

            with autocast("cuda"):
                action_logits, item_logits, temporal_logits = model(
                    events, items, deltas,
                    history=history,
                    tgt_key_padding_mask=pad_mask,
                    item_mask=item_valid,
                )
                val_action   += F.cross_entropy(action_logits[valid], tgt_actions[valid]).item()
                if item_valid.any():
                    val_item += F.cross_entropy(item_logits, tgt_items[item_valid]).item()
                val_temporal += F.cross_entropy(temporal_logits[valid], tgt_deltas[valid]).item()
            val_batches += 1

    val_loss = (val_action + val_item + val_temporal) / val_batches
    print(
        f"Val   {epoch}/{TRAIN_EPOCHS} | "
        f"action={val_action/val_batches:.4f} "
        f"item={val_item/val_batches:.4f} "
        f"temporal={val_temporal/val_batches:.4f} "
        f"| total={val_loss:.4f}"
    )

    if val_loss < best_loss:
        best_loss = val_loss
        model.save(MODEL_DIR / "session_transformer.pt")
        print(f"  Checkpoint saved (val_loss={best_loss:.4f})")

print(f"\nTraining complete. Best val loss: {best_loss:.4f}")
print(f"Model saved to: {MODEL_DIR / 'session_transformer.pt'}")

/home/mofu/code/thesis/pipeline/.venv/lib/python3.11/site-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


  [1/1] batch 8/8 | act=1.484 item=13.566 temp=3.963
Epoch 1/1 | action=1.4835 item=13.5662 temporal=3.9633 | total=19.0130 | 1.4s


Val   1/1 | action=1.3055 item=12.8298 temporal=3.5709 | total=17.7063


  SessionTransformer saved → /home/mofu/code/thesis/pipeline/output/models/session_transformer.pt
  Checkpoint saved (val_loss=17.7063)

Training complete. Best val loss: 17.7063
Model saved to: /home/mofu/code/thesis/pipeline/output/models/session_transformer.pt
